# Generate Fortran Code from YAML Configuration

This notebook demonstrates the complete workflow for generating Fortran source code from a YAML model configuration file.

## Overview

The BRNS (Biogeochemical Reaction Network Simulator) uses a Python-based code generation pipeline to convert high-level model descriptions (YAML format) into optimized Fortran source code.

**Key steps in this notebook:**
1. **Setup:** Load the ACGOrchestrator and discover available YAML models
2. **Configure:** Select a YAML model and output directory
3. **Generate:** Run the code generation pipeline (formula evaluation, ACG mapping, preprocessing, code generation)
4. **Inspect:** Review the generated Fortran files

## Next Steps

After successful code generation:
- Use the `build_python.sh` script to compile and run your generated Fortran code:
  ```bash
  ./build_python.sh -c /path/to/your/model.yaml -i /path/to/inputs
  ```
- Or manually build the generated code using `gfortran` with the files in `generated_fortran/<dir_name>/`
- Results will be saved as `.dat` files in the specified output directory

## Step 1: Setup and Discovery

First, we set up the project environment and discover available YAML models.

In [1]:
from pathlib import Path
import sys
from pprint import pprint

ROOT = Path.cwd().resolve()
if not (ROOT / 'acg_brns').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from acg_brns.acg_orchestrator import ACGOrchestrator

models_dir = ROOT / 'models'
available_models = sorted(path.name for path in models_dir.glob('*.yaml'))
available_models

['canfield.yaml',
 'canfield_refactored.yaml',
 'multiple_species_example.yaml',
 'single_species_example.yaml']

## Step 2: Configuration - Select Model and Output Directory

In the cell below, choose which YAML model to generate Fortran code for.
The `yaml_name` must exist in the `models/` folder (see available models above).
The `dir_name` determines where the generated Fortran files will be saved in `generated_fortran/`.

**Example models:**
- `single_species_example.yaml` → generates code for single-species kinetics
- `multiple_species_example.yaml` → generates code for multi-species reactions
- `canfield_equilibrium.yaml` → generates code for Canfield equilibrium model

In [2]:
# yaml_name = 'multiple_species_example.yaml'
# dir_name ="multiple_species"

yaml_name = 'single_species_example.yaml'
dir_name = "single_species"

# yaml_name = 'canfield_refactored.yaml'
# dir_name = "equilibrium"

yaml_path = models_dir / yaml_name
output_dir = ROOT / 'generated_fortran' / dir_name

print(f'YAML: {yaml_path}')
print(f'Output: {output_dir}')

YAML: /mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/models/single_species_example.yaml
Output: /mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species


## Step 3: Code Generation Pipeline

The ACGOrchestrator executes the following pipeline stages:

1. **Load & Validate:** Read and parse the YAML configuration
2. **Evaluate Formulas:** Process symbolic expressions and rate laws
3. **Map to ACG Structures:** Convert to internal ACG (Automatic Code Generation) data structures
4. **Preprocessing:** Apply Gaussian elimination and generate symbolic Jacobian
5. **Code Generation:** Produce optimized Fortran source files

The verbose output below shows progress through each stage. After generation, the output directory will contain all generated `.f90` and `.inc` files.

In [3]:
orchestrator = ACGOrchestrator(
    yaml_path=str(yaml_path),
    output_dir=str(output_dir),
    verbose=True,
)
summary = orchestrator.generate()
pprint(summary)

ACGOrchestrator initialized:
  YAML: /mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/models/single_species_example.yaml
  Output: /mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species

=== Phase 1: Loading Configuration ===
✓ Loaded configuration from single_species_example.yaml
  Species: 1
  Reactions: 1

=== Phase 2: Evaluating Formulas ===
✓ Evaluated 20 parameters

=== Phase 3: Mapping to ACG Structures ===
✓ Mapped structures:
  Variables: 1
    Dissolved: 1
    Solid: 0
  Reactions: 1
  Bio-parameters: 1

=== Phase 4: Pre-processing (p0-p10) ===
  Running Gaussian elimination...
p4: Building coefficient matrix from equations...
  Coefficient matrix: (1, 1)
Detecting conservation laws from coefficient matrix...
  Augmented matrix shape: (1, 2)
p5: Gaussian elimination (Maple-style: NO normalization, full elimination)...
  Column 0: using row 0 as pivot
  Pivot rows: [0]
  Matrix transformed (NO column permutation!)
p6: Extracting stoichiometri

## Step 4: Inspect Generated Files

List and review all generated Fortran files:

In [4]:
sorted(output_dir.glob('*'))

[PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/basic.f'),
 PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/biogeo.f'),
 PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/boundaries.f'),
 PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/common_geo.inc'),
 PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/initialcond.f'),
 PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/issolid.f'),
 PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/jacobian.f'),
 PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/molecular.f'),
 PosixPath('/mnt/c/Tecklenburg.J/work/thermooptiplan/BRNSPackage/generated_fortran/single_species/no

## Step 5: Build and Run with build_python.sh

Now that the Fortran code has been generated, you can compile and run it using the `build_python.sh` script.

**Usage:**
```bash
# Navigate to the project root
cd /path/to/BRNSPackage

# Run the build script with your YAML file and input directory
./build_python.sh -c ./models/single_species_example.yaml -i ./path/to/inputs [-o ./output_dir]
```
**What build_python.sh does:**
1. Compiles the Fortran code using gfortran
2. Runs the executable and collects results into `.dat` files

**After successful execution:**
- Output files will be in `build_output/<model_name>/results/`
- Use `plot_results.ipynb` to visualize the results